# STA437 Final Project — SDSS Stellar Classification
## Central Question
*How well can photometric measurements alone recover spectroscopic classifications of celestial objects, and what latent structure underlies the photometric feature space?*

**Dataset:** Sloan Digital Sky Survey (SDSS) DR14 — 10,000 celestial objects (stars, galaxies, quasars)  
**Key features:** Five photometric magnitudes (u, g, r, i, z) and spectroscopic redshift

---


## 0. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA, FactorAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.covariance import MinCovDet
from sklearn.metrics import adjusted_rand_score, confusion_matrix, classification_report
import os
import warnings
warnings.filterwarnings('ignore')

# Consistent plot style
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
COLORS = {'STAR': '#2196F3', 'GALAXY': '#FF5722', 'QSO': '#4CAF50'}
PALETTE = [COLORS['GALAXY'], COLORS['STAR'], COLORS['QSO']]
os.makedirs('figures', exist_ok=True)


In [ ]:
# Load data
df = pd.read_csv('sdss_data.csv')
print(f"Loaded {len(df)} objects, {df.shape[1]} variables")
print(f"Classes: {df['class'].value_counts().to_dict()}")

# Feature definitions
photo_cols   = ['u', 'g', 'r', 'i', 'z']
spectro_cols = ['redshift']
all_phys     = photo_cols + spectro_cols

# Derived colour indices (standard astronomical practice)
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']
color_cols = ['u_g', 'g_r', 'r_i', 'i_z']

label_map = {'GALAXY': 0, 'STAR': 1, 'QSO': 2}
y_num = df['class'].map(label_map).values
y     = df['class'].values
cls_names = ['GALAXY', 'STAR', 'QSO']

df[photo_cols + spectro_cols].describe().round(3)


## 1. Exploratory Data Analysis

In [ ]:
# FIGURE 1 — Class overview and magnitude distributions
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 6, figure=fig, hspace=0.45, wspace=0.35)

ax_pie = fig.add_subplot(gs[0, :2])
counts = df['class'].value_counts()[['GALAXY','STAR','QSO']]
bars = ax_pie.bar(counts.index, counts.values,
                  color=[COLORS[c] for c in counts.index], edgecolor='white', width=0.6)
ax_pie.set_ylabel('Count')
ax_pie.set_title('(A) Class distribution')
for bar, val in zip(bars, counts.values):
    ax_pie.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
                f'{val}\n({val/len(df)*100:.0f}%)', ha='center', va='bottom', fontsize=9)
ax_pie.set_ylim(0, 6200)

ax_rs = fig.add_subplot(gs[0, 2:])
for cls in ['STAR','GALAXY','QSO']:
    sub = df[df['class']==cls]['redshift']
    ax_rs.hist(sub.clip(lower=0), bins=60, alpha=0.65, color=COLORS[cls],
               label=cls, density=True)
ax_rs.set_xlabel('Redshift (clipped at 0)')
ax_rs.set_ylabel('Density')
ax_rs.set_title('(B) Redshift distributions by class')
ax_rs.legend()
ax_rs.set_xlim(-0.05, 0.8)

for k, col in enumerate(photo_cols):
    ax = fig.add_subplot(gs[1 + k//3, (k%3)*2:(k%3)*2+2])
    for cls in ['STAR','GALAXY','QSO']:
        sub = df[df['class']==cls][col]
        sub = sub[(sub > 10) & (sub < 26)]
        ax.hist(sub, bins=40, alpha=0.6, color=COLORS[cls], label=cls, density=True)
    ax.set_xlabel(f'{col} (mag)')
    ax.set_ylabel('Density')
    ax.set_title(f'({chr(67+k)}) {col} band')
    if k == 0:
        ax.legend(fontsize=8)

fig.suptitle('Figure 1. SDSS Dataset Overview: Class Distribution and Photometric Magnitudes',
             fontsize=12, fontweight='bold', y=1.01)
plt.savefig('figures/fig1_eda_overview.png')
plt.show()
print("Redshift summary by class:")
print(df.groupby('class')['redshift'].describe().round(4))


In [ ]:
# FIGURE 2 — Correlation matrix and colour-colour diagram
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

feat_df = df[all_phys + color_cols].copy()
feat_df.columns = ['u','g','r','i','z','z-shift','u–g','g–r','r–i','i–z']
corr = feat_df.corr()
sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size':8})
axes[0].set_title('(A) Feature correlation matrix')

np.random.seed(42)
idx = np.random.choice(len(df), 3000, replace=False)
sub = df.iloc[idx]
for cls in ['STAR','GALAXY','QSO']:
    mask2 = sub['class'] == cls
    axes[1].scatter(sub.loc[mask2,'u_g'], sub.loc[mask2,'g_r'],
                    s=8, alpha=0.5, color=COLORS[cls], label=cls)
axes[1].set_xlabel('u – g (mag)')
axes[1].set_ylabel('g – r (mag)')
axes[1].set_xlim(-1, 5)
axes[1].set_ylim(-1, 3)
axes[1].legend(markerscale=3)
axes[1].set_title('(B) Colour–colour diagram (u–g vs g–r)')

fig.suptitle('Figure 2. Feature Correlations and Colour-Colour Diagram',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig2_correlations.png')
plt.show()


## 2. Multivariate Normality Assessment

In [ ]:
# Mardia's test implementation
def mardia_test(X):
    n, p = X.shape
    mu   = X.mean(axis=0)
    S    = np.cov(X, rowvar=False)
    Sinv = np.linalg.pinv(S)
    diff = X - mu
    D2   = (diff @ Sinv * diff).sum(axis=1)
    b1p  = ((diff @ Sinv @ diff.T)**3).mean()
    k_skew = n * b1p / 6
    p_skew = 1 - chi2.cdf(k_skew, df=p*(p+1)*(p+2)/6)
    b2p    = (D2**2).mean()
    mu_k   = p*(p+2)
    sigma_k = np.sqrt(8*p*(p+2)/n)
    z_kurt  = (b2p - mu_k) / sigma_k
    p_kurt  = 2*(1 - stats.norm.cdf(abs(z_kurt)))
    return D2, k_skew, p_skew, b2p, z_kurt, p_kurt

X_photo = df[photo_cols].values
X_photo_std = StandardScaler().fit_transform(X_photo)

print("Mardia's test on full dataset (photometric features):")
D2_all, ks, ps, b2, zk, pk = mardia_test(X_photo_std)
print(f"  Skewness statistic: {ks:.2f},  p = {ps:.4e}")
print(f"  Kurtosis z-score:   {zk:.2f},  p = {pk:.4e}")
print()
print("Per-class Mardia's test:")
for cls in ['GALAXY','STAR','QSO']:
    Xc = X_photo_std[y == cls]
    _, ks_c, ps_c, _, zk_c, pk_c = mardia_test(Xc)
    print(f"  {cls:8s}  skew p={ps_c:.3e}  kurt p={pk_c:.3e}")


In [ ]:
# FIGURE 3 — Mahalanobis distance QQ plots
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
p = X_photo_std.shape[1]
chi2_quantiles = chi2.ppf(np.linspace(0.005, 0.995, len(X_photo_std)), df=p)
D2_photo, *_ = mardia_test(X_photo_std)

axes[0].plot(np.sort(chi2_quantiles), np.sort(D2_photo), '.', markersize=2, alpha=0.4, color='steelblue')
axes[0].plot([0, chi2_quantiles.max()], [0, chi2_quantiles.max()], 'r--', lw=1.5)
axes[0].set_title('All objects (n=10,000)')
axes[0].set_xlabel('χ²(5) quantiles')
axes[0].set_ylabel('Mahalanobis D²')

for k, cls in enumerate(['GALAXY','STAR','QSO']):
    Xc  = X_photo_std[y == cls]
    nc  = len(Xc)
    D2c, *_ = mardia_test(Xc)
    chi2q = chi2.ppf(np.linspace(0.005, 0.995, nc), df=p)
    axes[k+1].plot(np.sort(chi2q), np.sort(D2c), '.', markersize=2, alpha=0.4, color=COLORS[cls])
    axes[k+1].plot([0, chi2q.max()], [0, chi2q.max()], 'r--', lw=1.5)
    axes[k+1].set_title(f'{cls} (n={nc})')
    axes[k+1].set_xlabel('χ²(5) quantiles')

fig.suptitle('Figure 3. Mahalanobis Distance Q–Q Plots (photometric features)\n'
             'Deviation from diagonal indicates departure from multivariate normality.',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig3_mvn_qqplot.png')
plt.show()


## 3. Principal Component Analysis

In [ ]:
# FIGURE 4 — PCA on photometric + redshift features
X_std_full = StandardScaler().fit_transform(df[all_phys].values)
pca = PCA()
scores_pca = pca.fit_transform(X_std_full)

print("PCA explained variance:")
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.3f}  (cumulative: {pca.explained_variance_ratio_[:i+1].sum():.3f})")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
cumvar = np.cumsum(pca.explained_variance_ratio_)
ax.bar(range(1, 7), pca.explained_variance_ratio_*100, color='steelblue', alpha=0.8)
ax.plot(range(1, 7), cumvar*100, 'o-', color='darkorange', lw=2, label='Cumulative')
ax.axhline(90, ls='--', color='grey', lw=1, alpha=0.7, label='90% threshold')
ax.set_xlabel('Principal component')
ax.set_ylabel('Explained variance (%)')
ax.set_title('(A) Scree plot')
ax.legend()
ax.set_xticks(range(1, 7))

np.random.seed(0)
idx2 = np.random.choice(len(scores_pca), 3000, replace=False)
ax = axes[1]
for cls in ['GALAXY','STAR','QSO']:
    m = y[idx2] == cls
    ax.scatter(scores_pca[idx2][m, 0], scores_pca[idx2][m, 1],
               s=8, alpha=0.5, color=COLORS[cls], label=cls)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('(B) PC1 vs PC2 (n=3,000)')
ax.legend(markerscale=3)

ax = axes[2]
loadings = pd.DataFrame(pca.components_.T,
                        index=all_phys,
                        columns=[f'PC{i+1}' for i in range(len(all_phys))])
sns.heatmap(loadings, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size':9})
ax.set_title('(C) PCA loadings')

fig.suptitle('Figure 4. Principal Component Analysis (photometric magnitudes + redshift)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig4_pca.png')
plt.show()


## 4. Factor Analysis

In [ ]:
# FIGURE 5 — Factor Analysis on photometric features
X_photo_std2 = StandardScaler().fit_transform(df[photo_cols].values)

ll_scores = []
for nf in range(1, 5):
    fa = FactorAnalysis(n_components=nf, random_state=42, max_iter=1000)
    fa.fit(X_photo_std2)
    ll_scores.append(fa.score(X_photo_std2))

fa2 = FactorAnalysis(n_components=2, rotation='varimax', random_state=42, max_iter=1000)
fa2.fit(X_photo_std2)
fa_scores2 = fa2.transform(X_photo_std2)
loadings_fa = pd.DataFrame(fa2.components_.T,
                            index=photo_cols, columns=['Factor 1', 'Factor 2'])
print("Factor loadings (2-factor varimax):")
print(loadings_fa.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ax = axes[0]
ax.plot(range(1, 5), ll_scores, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Number of factors')
ax.set_ylabel('Log-likelihood (per sample)')
ax.set_title('(A) Model fit vs. number of factors')
ax.set_xticks(range(1, 5))

ax = axes[1]
sns.heatmap(loadings_fa, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size':11})
ax.set_title('(B) Factor loadings (varimax)\n2-factor model')

ax = axes[2]
np.random.seed(1)
idx3 = np.random.choice(len(fa_scores2), 3000, replace=False)
for cls in ['GALAXY','STAR','QSO']:
    m = y[idx3] == cls
    ax.scatter(fa_scores2[idx3][m, 0], fa_scores2[idx3][m, 1],
               s=8, alpha=0.5, color=COLORS[cls], label=cls)
ax.set_xlabel('Factor 1 score')
ax.set_ylabel('Factor 2 score')
ax.set_title('(C) Factor scores by class (n=3,000)')
ax.legend(markerscale=3)

fig.suptitle('Figure 5. Factor Analysis of Photometric Magnitudes (varimax rotation)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig5_factor_analysis.png')
plt.show()


## 5. Clustering: GMM and k-means

In [ ]:
# FIGURE 6 — GMM clustering: photo only vs photo + redshift
X_photo_std3 = StandardScaler().fit_transform(df[photo_cols].values)
pca_photo = PCA(n_components=2).fit(X_photo_std3)
Z_photo   = pca_photo.transform(X_photo_std3)

X_full_std = StandardScaler().fit_transform(df[all_phys].values)
pca_full   = PCA(n_components=2).fit(X_full_std)
Z_full     = pca_full.transform(X_full_std)

gmm_photo = GaussianMixture(n_components=3, covariance_type='full', n_init=5,
                             random_state=42).fit(X_photo_std3)
gmm_full  = GaussianMixture(n_components=3, covariance_type='full', n_init=5,
                              random_state=42).fit(X_full_std)
labels_gmm_photo = gmm_photo.predict(X_photo_std3)
labels_gmm_full  = gmm_full.predict(X_full_std)

from itertools import permutations
def align_labels(pred, true_num):
    best_perm, best_ari = None, -1
    for perm in permutations([0,1,2]):
        remapped = np.vectorize(dict(zip([0,1,2], perm)).get)(pred)
        ari = adjusted_rand_score(true_num, remapped)
        if ari > best_ari:
            best_ari = ari; best_perm = perm
    return np.vectorize(dict(zip([0,1,2], best_perm)).get)(pred), best_ari

labels_aligned_photo, ari_photo = align_labels(labels_gmm_photo, y_num)
labels_aligned_full,  ari_full  = align_labels(labels_gmm_full,  y_num)
print(f"GMM ARI — photo only: {ari_photo:.3f}  |  photo+redshift: {ari_full:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
np.random.seed(2)
idx4 = np.random.choice(len(Z_photo), 3000, replace=False)

for cls in cls_names:
    m = y[idx4] == cls
    axes[0].scatter(Z_photo[idx4][m,0], Z_photo[idx4][m,1],
                    s=8, alpha=0.4, color=COLORS[cls], label=cls)
axes[0].set_title('(A) True labels (photo PCA space)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=3)

for ci in range(3):
    m = labels_aligned_photo[idx4] == ci
    axes[1].scatter(Z_photo[idx4][m,0], Z_photo[idx4][m,1],
                    s=8, alpha=0.4, color=PALETTE[ci], label=cls_names[ci])
axes[1].set_title(f'(B) GMM — photo only\nARI = {ari_photo:.3f}')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=3)

for ci in range(3):
    m = labels_aligned_full[idx4] == ci
    axes[2].scatter(Z_full[idx4][m,0], Z_full[idx4][m,1],
                    s=8, alpha=0.4, color=PALETTE[ci], label=cls_names[ci])
axes[2].set_title(f'(C) GMM — photo + redshift\nARI = {ari_full:.3f}')
axes[2].set_xlabel('PC1 (with redshift)'); axes[2].set_ylabel('PC2')
axes[2].legend(markerscale=3)

fig.suptitle('Figure 6. GMM Clustering: Photometry Alone vs. Photometry + Redshift',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig6_gmm_clustering.png')
plt.show()


## 6. Linear Discriminant Analysis and Cross-Validation

In [ ]:
# FIGURE 7 — LDA + 5-fold cross-validation
Xp = StandardScaler().fit_transform(df[photo_cols].values)
Xa = StandardScaler().fit_transform(df[all_phys].values)

lda_p = LinearDiscriminantAnalysis().fit(Xp, y)
lda_a = LinearDiscriminantAnalysis().fit(Xa, y)
Z_lda_p = lda_p.transform(Xp)
Z_lda_a = lda_a.transform(Xa)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_lda_p = cross_val_score(LinearDiscriminantAnalysis(), Xp, y, cv=cv, scoring='accuracy')
acc_lda_a = cross_val_score(LinearDiscriminantAnalysis(), Xa, y, cv=cv, scoring='accuracy')
acc_lr_p  = cross_val_score(LogisticRegression(max_iter=500, C=1.0, random_state=42),
                             Xp, y, cv=cv, scoring='accuracy')
acc_lr_a  = cross_val_score(LogisticRegression(max_iter=500, C=1.0, random_state=42),
                             Xa, y, cv=cv, scoring='accuracy')

print("5-fold CV accuracy:")
print(f"  LDA (photo only):      {acc_lda_p.mean():.3f} ± {acc_lda_p.std():.3f}")
print(f"  LDA (photo+redshift):  {acc_lda_a.mean():.3f} ± {acc_lda_a.std():.3f}")
print(f"  LR  (photo only):      {acc_lr_p.mean():.3f}  ± {acc_lr_p.std():.3f}")
print(f"  LR  (photo+redshift):  {acc_lr_a.mean():.3f}  ± {acc_lr_a.std():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
np.random.seed(3)
idx5 = np.random.choice(len(Z_lda_a), 3000, replace=False)

for cls in cls_names:
    m = y[idx5] == cls
    axes[0].scatter(Z_lda_p[idx5][m,0], Z_lda_p[idx5][m,1],
                    s=8, alpha=0.45, color=COLORS[cls], label=cls)
axes[0].set_xlabel('LD1'); axes[0].set_ylabel('LD2')
axes[0].set_title(f'(A) LDA — photo only\nCV acc = {acc_lda_p.mean():.3f}')
axes[0].legend(markerscale=3)

for cls in cls_names:
    m = y[idx5] == cls
    axes[1].scatter(Z_lda_a[idx5][m,0], Z_lda_a[idx5][m,1],
                    s=8, alpha=0.45, color=COLORS[cls], label=cls)
axes[1].set_xlabel('LD1'); axes[1].set_ylabel('LD2')
axes[1].set_title(f'(B) LDA — photo + redshift\nCV acc = {acc_lda_a.mean():.3f}')
axes[1].legend(markerscale=3)

methods = ['LDA\n(photo)', 'LDA\n(photo+z)', 'LR\n(photo)', 'LR\n(photo+z)']
means   = [acc_lda_p.mean(), acc_lda_a.mean(), acc_lr_p.mean(), acc_lr_a.mean()]
stds    = [acc_lda_p.std(),  acc_lda_a.std(),  acc_lr_p.std(),  acc_lr_a.std()]
clrs    = ['#90CAF9','#1565C0','#FFAB91','#BF360C']
bars    = axes[2].bar(methods, means, yerr=stds, color=clrs, capsize=5, edgecolor='white', width=0.5)
axes[2].set_ylim(0.5, 1.05)
axes[2].set_ylabel('5-fold CV accuracy')
axes[2].set_title('(C) Classification accuracy\n(error bars = ±1 SD across folds)')
for bar, mean in zip(bars, means):
    axes[2].text(bar.get_x() + bar.get_width()/2, mean + 0.01, f'{mean:.3f}',
                 ha='center', va='bottom', fontsize=9)

fig.suptitle('Figure 7. Linear Discriminant Analysis and Cross-Validated Classification Accuracy',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig7_lda_cv.png')
plt.show()

print("\nClassification report — LR (photo+redshift):")
lr_full = LogisticRegression(max_iter=500, C=1.0, random_state=42).fit(Xa, y)
print(classification_report(y, lr_full.predict(Xa), target_names=cls_names))


## 7. Extensions: Robust Estimation and Kernel PCA

In [ ]:
# FIGURE 8 — Robust PCA (MCD) + Kernel PCA
Xp_std = StandardScaler().fit_transform(df[photo_cols].values)

mcd = MinCovDet(support_fraction=0.85, random_state=42)
mcd.fit(Xp_std)
mu_robust = mcd.location_
S_robust  = mcd.covariance_

# Classical Mahalanobis
mu_c = Xp_std.mean(axis=0)
S_c  = np.cov(Xp_std, rowvar=False)
diff_c = Xp_std - mu_c
maha_classic = np.sqrt(np.einsum('ij,jk,ik->i', diff_c, np.linalg.pinv(S_c), diff_c))
diff_r = Xp_std - mu_robust
maha_robust = np.sqrt(np.einsum('ij,jk,ik->i', diff_r, np.linalg.pinv(S_robust), diff_r))

thresh_robust = np.sqrt(chi2.ppf(0.975, df=len(photo_cols)))
outlier_mask  = maha_robust > thresh_robust
n_outliers    = outlier_mask.sum()
print(f"Robust outliers: {n_outliers} ({n_outliers/len(Xp_std)*100:.1f}%)")
print("Outlier class breakdown:", pd.Series(y[outlier_mask]).value_counts().to_dict())

# Robust PCA
eigvals, eigvecs = np.linalg.eigh(S_robust)
order = np.argsort(eigvals)[::-1]
eigvecs = eigvecs[:, order]
Z_robust_pca = (Xp_std - mu_robust) @ eigvecs[:, :2]

# Kernel PCA
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.5, random_state=42)
Z_kpca = kpca.fit_transform(Xp_std)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
np.random.seed(4)

not_out = ~outlier_mask
np.random.seed(42)
subsample_mask = not_out & (np.random.rand(len(Xp_std)) < 0.3)
axes[0].scatter(maha_classic[subsample_mask], maha_robust[subsample_mask],
                s=5, alpha=0.3, color='steelblue', label='Inlier')
axes[0].scatter(maha_classic[outlier_mask], maha_robust[outlier_mask],
                s=20, alpha=0.7, color='crimson', label=f'Outlier (n={n_outliers})', zorder=5)
axes[0].axhline(thresh_robust, color='crimson', ls='--', lw=1.5,
                label=f'Robust threshold ({thresh_robust:.2f})')
axes[0].set_xlabel('Classical Mahalanobis distance')
axes[0].set_ylabel('Robust (MCD) Mahalanobis distance')
axes[0].set_title('(A) Robust outlier detection')
axes[0].legend(fontsize=8)

idx6 = np.random.choice(len(Xp_std), 3000, replace=False)
for cls in cls_names:
    m = y[idx6] == cls
    axes[1].scatter(Z_robust_pca[idx6][m,0], Z_robust_pca[idx6][m,1],
                    s=8, alpha=0.45, color=COLORS[cls], label=cls)
axes[1].set_xlabel('Robust PC1'); axes[1].set_ylabel('Robust PC2')
axes[1].set_title('(B) Robust PCA projection (MCD)')
axes[1].legend(markerscale=3)

for cls in cls_names:
    m = y[idx6] == cls
    axes[2].scatter(Z_kpca[idx6][m,0], Z_kpca[idx6][m,1],
                    s=8, alpha=0.45, color=COLORS[cls], label=cls)
axes[2].set_xlabel('Kernel PC1 (RBF)'); axes[2].set_ylabel('Kernel PC2 (RBF)')
axes[2].set_title('(C) Kernel PCA (RBF, γ=0.5)')
axes[2].legend(markerscale=3)

fig.suptitle('Figure 8. Extensions: Robust Estimation (MCD) and Kernel PCA',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig8_robust_kernel.png')
plt.show()


## 8. Summary

| Method | Key Finding |
|--------|-------------|
| EDA | Redshift is near-perfectly discriminative; photometric magnitudes overlap substantially between classes |
| MVN | All classes reject multivariate normality (Mardia's test); QSOs most non-normal due to high-redshift outliers |
| PCA | PC1 (76.3%) captures overall brightness; PC2 (14.6%) captures blue-end vs red-end colour gradient |
| Factor Analysis | Two factors: "infrared brightness" (r/i/z) and "UV excess" (u/g); loadings support standard astronomical colour indices |
| GMM clustering | ARI = 0.16 (photo only) vs ARI = 0.91 (photo + redshift) — redshift is critical for clustering |
| LDA | Photo only: 92.0% CV accuracy; photo + redshift: 92.0% (LDA cannot exploit nonlinear redshift distribution) |
| Logistic Regression | Photo only: 93.3%; photo + redshift: 97.1% — LR exploits redshift effectively |
| Robust PCA | 19.7% of objects flagged as outliers by MCD; QSOs overrepresented (96.0% of QSOs are outliers) |
| Kernel PCA | Nonlinear structure revealed: QSOs form a curved manifold in RBF kernel space |

**Central finding:** Photometric data alone achieves ~93% classification accuracy (LR), but spectroscopic redshift pushes this to ~97% and is essential for unsupervised recovery of the three object classes (ARI jumps from 0.16 to 0.91).
